# L11c: can a tool workflow terminate and still be wrong?

Read the connected lesson and predict before running. No API key or hosted provider is used. All original fixture data is MIT. This notebook imports the shared tested source; no copied model algorithm.


## Execution contract

Run `uv sync --locked --project examples/applications/workflows`, select that environment's Python kernel, and start in this project or its notebooks directory. Pinned Q8 SmolLM2 GGUF: 144,811,360-byte first download, SHA-256 checked, CPU computation, native context 768, one thread. Installation compiles llama-cpp-python and needs a compiler/CMake. Package/build downloads are additional.

On the author host each script took about 1–10 seconds after setup; hardware and prefix caching affect timing. Recorded child process memory is roughly 0.5–0.6 GiB, not a precision-comparison benchmark. On macOS the native build initializes Metal even with CPU buffers; use a normal host runtime if a managed sandbox denies initialization. See README for exact pins and provenance. Restart the kernel and Run All; the first run needs network for the public artifact, later runs can set HF_HUB_OFFLINE=1.


## Inspect data before inference

These original tasks have stable IDs. Arithmetic has 4 design / 8 evaluation cases; extraction has 3 design / 9 evaluation cases. Memory/search/tools are declared mechanism probes. No training or evaluation-set tuning occurs. Demonstrations use design IDs only; gold answers are not inserted into per-case decoding domains.


In [ ]:
import sys, json
from pathlib import Path
project = Path.cwd()
if project.name == "notebooks": project = project.parent
root = project.parents[2]
sys.path.insert(0, str(project / "src"))
fixture = json.loads((root / "examples/data/fixtures/workflow-tasks.json").read_text())
print("Original fixture fields:", list(fixture))
for name in ["arithmetic", "extraction"]:
    rows = fixture[name]
    print(name, {split: sum(r["split"] == split for r in rows) for split in ["design", "evaluation"]})
    print("Design example:", next(r for r in rows if r["split"] == "design"))


## Predict and inspect the baseline

Inspect the finite tool and final-value domains. Predict how invalid booleans, repeated actions, and a wrong tool choice should appear. Scripted/FakeListLLM tests measure mechanics, separately from actual GGUF choices.

Read the module source linked in the lesson before invoking it. It includes the baseline and declared alternatives in one bounded run. A result need not improve.


In [ ]:
from atlas_workflows.agents import main
main()

## Evaluate and inspect failures

Read the actual artifact produced above. Shape validity, semantic correctness, retained evidence, and controller termination answer different questions. Raw calls retain messages, native token counts, output caps, stop reasons, and unedited output. The tiny model's failures remain part of the evidence.


In [ ]:
artifact_path = root / 'examples/artifacts/e5-agents/results.json'
result = json.loads(artifact_path.read_text())
print(json.dumps([{'id':r['id'],'engine':r['engine'],'expected':r['expected'],'correct':r['correct'],'outcome':r['outcome']} for r in result['real_cases']], indent=2))
print('Fake-model stop reasons:', [(r['id'], r['framework']['stop_reason']) for r in result['fake_cases']])

## Work and controlled comparisons

Prompt/input and generated/output tokens are recorded per call. Summary updates, repeated samples, and model-guided ranking add calls. Single sequential wall timings reuse a loaded model and prefix cache; do not treat them as a speed leaderboard, paid cost, or causal comparison across unrelated paths. The declared comparison changes only its named component except where explicitly labeled as a separate mechanism probe.


In [ ]:
# Reload the saved result, preserving its link to actual inputs/settings.
reloaded = json.loads(artifact_path.read_text())
assert reloaded == result
print('Saved artifact reload checked:', artifact_path.name)
print('Model artifact revision:', result['model']['revision'])

## Practice and transfer

Does max_steps count as task success? Does a valid action ID prove the chosen tool is appropriate?

<details><summary>Hint and solution rationale</summary>

Neither establishes an answer. The controller guarantees a bounded path, while source-based correctness and tool selection are separately evaluated. Operands are predeclared; this is a restricted controller.

</details>

For a transfer experiment, predeclare a new design/evaluation fixture and change one component. Do not select a best prompt or seed on these evaluation outcomes. Report: task IDs; baseline; changed component; expected failure; raw evidence; parser/verifier; denominators; tokens/calls; observed limitations. Broader model, device, or unrestricted-agent claims need new evidence.


## Verification boundary

The notebook and CLI call the same source. Run notebook verification before final `--record` CLI runs because timing-bearing artifacts are overwritten. Then sync published results. Execution and author review do not establish independent teaching acceptance or beginner usability. No source PDF or copied artwork is distributed.
